In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Importing data_file by parsing data_type in columns
df = pd.read_csv(r'C:visits_and_wait_times_m.csv', 
                 parse_dates = ['visit_date', 'check_in_time', 'seen_by_provider_time'])

In [3]:
print(df.shape)
print(df.dtypes)

(20000, 9)
visit_id                            str
patient_id                          str
visit_date               datetime64[us]
visit_type                          str
department                          str
check_in_time            datetime64[us]
seen_by_provider_time    datetime64[us]
hcahps_score                    float64
provider_id                         str
dtype: object


In [4]:
df.head(10)

,visit_id,patient_id,visit_date,visit_type,department,check_in_time,seen_by_provider_time,hcahps_score,provider_id
0,VIS-000001,PAT-01825,2017-05-21,ER,Pediatrics,2017-05-21 09:47:00,2017-05-21 10:41:00,76.5,PRV-0140
1,VIS-000002,PAT-01425,2021-01-17,ER,Oncology,2021-01-17 13:01:00,2021-01-17 14:42:00,81.8,PRV-0144
2,VIS-000003,PAT-03258,2022-12-26,Outpatient,Oncology,2022-12-26 14:51:00,2022-12-26 15:13:00,69.2,PRV-0002
3,VIS-000004,PAT-02616,2022-06-24,Outpatient,Pediatrics,2022-06-24 16:06:00,2022-06-24 16:52:00,81.2,PRV-0024
4,VIS-000005,PAT-06225,2018-06-26,Outpatient,Pulmonology,2018-06-26 16:34:00,2018-06-26 16:54:00,84.0,PRV-0032
5,VIS-000006,PAT-06202,2017-05-31,Follow-up,General,2017-05-31 18:36:00,2017-05-31 18:50:00,72.9,PRV-0050
6,VIS-000007,PAT-01140,2017-04-27,Follow-up,Psychiatry,2017-04-27 18:55:00,2017-04-27 19:12:00,57.0,PRV-0026
7,VIS-000008,PAT-06228,2019-11-22,Follow-up,Pediatrics,2019-11-22 11:42:00,2019-11-22 11:51:00,71.8,PRV-0069
8,VIS-000009,PAT-01170,2021-03-29,Outpatient,Neurology,2021-03-29 12:17:00,2021-03-29 12:33:00,71.1,PRV-0164
9,VIS-000010,PAT-09126,2019-06-16,Follow-up,Pulmonology,2019-06-16 10:02:00,2019-06-16 10:14:00,62.5,PRV-0081


In [5]:
# Check integrity of hcahps_score col
print(df['hcahps_score'].describe())
print(df[(df['hcahps_score'] < 0) | (df['hcahps_score'] > 100)].shape)
print(df['hcahps_score'].isna().sum())

count    20000.000000
mean        73.622560
std         57.136224
min        -15.500000
25%         61.600000
50%         72.100000
75%         81.400000
max        999.500000
Name: hcahps_score, dtype: float64
(799, 9)
0


In [6]:
# Remove unvalid scores
df['hcahps_score'] = df['hcahps_score'].mask((df['hcahps_score'] < 0) | (df['hcahps_score'] > 100))

In [7]:
print(df['hcahps_score'].describe())
print(df[(df['hcahps_score'] < 0) | (df['hcahps_score'] > 100)].shape)
print(df['hcahps_score'].isna().sum())

count    19201.000000
mean        70.971731
std         14.127392
min          0.000000
25%         62.000000
50%         72.000000
75%         81.000000
max        100.000000
Name: hcahps_score, dtype: float64
(0, 9)
799


In [8]:
# Check the logic between two date intervals
invalid_date = df.query("seen_by_provider_time < check_in_time")
print(invalid_date.shape)
invalid_date.head()

(1000, 9)


,visit_id,patient_id,visit_date,visit_type,department,check_in_time,seen_by_provider_time,hcahps_score,provider_id
23,VIS-000024,PAT-09076,2019-04-23,ER,Maternity,2019-04-23 08:04:00,2019-04-23 06:39:00,89.6,PRV-0009
47,VIS-000048,PAT-00407,2018-06-20,Follow-up,Orthopedics,2018-06-20 15:47:00,2018-06-20 15:14:00,61.9,PRV-0064
50,VIS-000051,PAT-04574,2020-09-17,ER,Psychiatry,2020-09-17 13:01:00,2020-09-17 11:02:00,56.1,PRV-0030
51,VIS-000052,PAT-04280,2018-05-16,ER,General,2018-05-16 11:50:00,2018-05-16 11:38:00,69.2,PRV-0081
71,VIS-000072,PAT-08819,2022-10-07,Follow-up,Dermatology,2022-10-07 13:17:00,2022-10-07 12:43:00,74.2,PRV-0112


In [9]:
# Masking all invalid dates
mask = df.query("seen_by_provider_time < check_in_time").index
mask[:10]

Index([23, 47, 50, 51, 71, 74, 89, 93, 111, 142], dtype='int64')

In [10]:
# Update seen_by_provider_time date 
df.loc[mask, 'seen_by_provider_time'] = np.nan

In [12]:
df.loc[mask, ['visit_id', 'visit_date', 'check_in_time', 'seen_by_provider_time']].head(10)

,visit_id,visit_date,check_in_time,seen_by_provider_time
23,VIS-000024,2019-04-23,2019-04-23 08:04:00,NaT
47,VIS-000048,2018-06-20,2018-06-20 15:47:00,NaT
50,VIS-000051,2020-09-17,2020-09-17 13:01:00,NaT
51,VIS-000052,2018-05-16,2018-05-16 11:50:00,NaT
71,VIS-000072,2022-10-07,2022-10-07 13:17:00,NaT
74,VIS-000075,2018-08-25,2018-08-25 11:29:00,NaT
89,VIS-000090,2021-09-09,2021-09-09 11:25:00,NaT
93,VIS-000094,2021-02-21,2021-02-21 11:19:00,NaT
111,VIS-000112,2018-07-26,2018-07-26 06:55:00,NaT
142,VIS-000143,2020-11-13,2020-11-13 19:14:00,NaT


In [13]:
df.to_csv("visits_and_wait_time_cl.csv", index=False)